In [48]:
import pandas as pd
import numpy as np
import itertools
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier,LogisticRegression
from sklearn.naive_bayes import MultinomialNB,GaussianNB,ComplementNB,BernoulliNB,CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

In [2]:
dataset=pd.read_csv("whatsapp_chat_labeled.csv")
dataset

,Date,Time,Name,Chat,Label
0,10/14/26,1:47 PM,Nivetha,Good morning,Normal
1,10/14/26,1:47 PM,Nivetha,How are you?,Normal
2,10/14/26,1:49 PM,Kirthi,Hi,Normal
3,10/14/26,1:50 PM,Kirthi,Good..u?,Normal
4,10/14/26,1:50 PM,Nivetha,Fine😊,Normal
...,...,...,...,...,...
144,10/19/26,12:45 PM,Kirthi,"Sure, come over and I'll give you an empty bowl😂",Normal
145,10/19/26,12:45 PM,Nivetha,"Wow, such hospitality! I deserve better from you.",Flirty
146,10/19/26,12:46 PM,Kirthi,May be you do. I'll consider it😜,Flirty
147,10/19/26,12:46 PM,Nivetha,"I'll remember that. Anyway, I need to get back...",Normal


In [3]:
x=dataset["Chat"]
y=dataset["Label"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=53)

In [5]:
count_vectorizer = CountVectorizer(stop_words='english')
count_train = count_vectorizer.fit_transform(X_train)
count_test = count_vectorizer.transform(X_test)
print("Unique_words:",len(count_vectorizer.get_feature_names_out()))#unique words
print(count_train.toarray())

Unique_words: 155
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]]


In [50]:
param_grid = {'alpha':[0.1,0.01,1.0,5.0]}
grid = GridSearchCV(MultinomialNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
The f1_macro value for best parameter {'alpha': 0.01}: 0.5798319327731092
The confusion Matrix:
 [[14 16]
 [ 5 15]]
The report:
               precision    recall  f1-score   support

      Flirty       0.74      0.47      0.57        30
      Normal       0.48      0.75      0.59        20

    accuracy                           0.58        50
   macro avg       0.61      0.61      0.58        50
weighted avg       0.64      0.58      0.58        50



In [74]:
param_grid = {'C':[0.1,0.01,1.0],'loss':["hinge","squared_hinge"]}
grid = GridSearchCV(PassiveAggressiveClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
The f1_macro value for best parameter {'C': 0.1, 'loss': 'hinge'}: 0.5784825371336813
The confusion Matrix:
 [[13 17]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.76      0.43      0.55        30
      Normal       0.48      0.80      0.60        20

    accuracy                           0.58        50
   macro avg       0.62      0.62      0.58        50
weighted avg       0.65      0.58      0.57        50



In [86]:
param_grid = {'C':[0.1,0.01,1.0],'penalty':['l2'],'solver':['lbfgs','liblinear','newton-cg','newton-cholesky', 'sag', 'saga']}
grid = GridSearchCV(LogisticRegression(random_state=0,max_iter=1000), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
The f1_macro value for best parameter {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}: 0.5353535353535354
The confusion Matrix:
 [[11 19]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.73      0.37      0.49        30
      Normal       0.46      0.80      0.58        20

    accuracy                           0.54        50
   macro avg       0.60      0.58      0.54        50
weighted avg       0.62      0.54      0.53        50



In [102]:
param_grid = {'C':[0.1,0.01,1.0],'kernel':['linear','poly','rbf','sigmoid'],'gamma':['scale','auto']}
grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
The f1_macro value for best parameter {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}: 0.5169082125603865
The confusion Matrix:
 [[11 19]
 [ 5 15]]
The report:
               precision    recall  f1-score   support

      Flirty       0.69      0.37      0.48        30
      Normal       0.44      0.75      0.56        20

    accuracy                           0.52        50
   macro avg       0.56      0.56      0.52        50
weighted avg       0.59      0.52      0.51        50



In [114]:
param_grid = {'n_estimators':[10,50,100],'criterion':['gini','entropy','log_loss'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced','balanced_subsample']}
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 108 candidates, totalling 324 fits
The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_split': 4, 'n_estimators': 10}: 0.4949494949494949
The confusion Matrix:
 [[10 20]
 [ 5 15]]
The report:
               precision    recall  f1-score   support

      Flirty       0.67      0.33      0.44        30
      Normal       0.43      0.75      0.55        20

    accuracy                           0.50        50
   macro avg       0.55      0.54      0.49        50
weighted avg       0.57      0.50      0.48        50



In [124]:
param_grid = {'criterion':['gini','entropy','log_loss'],'splitter':['best','random'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced']}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'gini', 'max_features': 'log2', 'min_samples_split': 3, 'splitter': 'random'}: 0.4663382594417077
The confusion Matrix:
 [[ 8 22]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.67      0.27      0.38        30
      Normal       0.42      0.80      0.55        20

    accuracy                           0.48        50
   macro avg       0.54      0.53      0.47        50
weighted avg       0.57      0.48      0.45        50



In [126]:
param_grid = {'n_neighbors':[2,3,4,5],'weights':['uniform','distance'],'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(count_train, y_train) 
grid_predictions = grid.predict(count_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 64 candidates, totalling 192 fits
The f1_macro value for best parameter {'algorithm': 'auto', 'n_neighbors': 4, 'p': 1, 'weights': 'uniform'}: 0.31066176470588236
The confusion Matrix:
 [[ 1 29]
 [ 1 19]]
The report:
               precision    recall  f1-score   support

      Flirty       0.50      0.03      0.06        30
      Normal       0.40      0.95      0.56        20

    accuracy                           0.40        50
   macro avg       0.45      0.49      0.31        50
weighted avg       0.46      0.40      0.26        50

